In [117]:
import requests
import geopandas as gpd
import json
from shapely.geometry import shape

# URL запроса
url = "https://nspd.gov.ru/api/aeggis/v4/36048/wms"
params = {
    "REQUEST": "GetFeatureInfo",
    "QUERY_LAYERS": "36048",
    "SERVICE": "WMS",
    "VERSION": "1.3.0",
    "FORMAT": "image/png",
    "STYLES": "",
    "TRANSPARENT": "true",
    "LAYERS": "36048",
    "RANDOM": "0.12828180732088468",
    "INFO_FORMAT": "application/json",
    "FEATURE_COUNT": "10",
    "I": "201",
    "J": "316",
    "WIDTH": "512",
    "HEIGHT": "512",
    "CRS": "EPSG:3857",
    "BBOX": "3130860.6785608195,8140237.7642581295,3757032.8142729835,8766409.899970293"
}



# Отправка запроса
response = requests.get(url, params=params, verify=False)
data = response.json()

# Преобразование в GeoDataFrame
features = data.get("features", [])
geoms = [shape(f["geometry"]) for f in features]
attrs = [f["properties"] for f in features]

gdf = gpd.GeoDataFrame(attrs, geometry=geoms, crs="EPSG:3857")

# Преобразуем в WGS84, если нужно
#gdf = gdf.to_crs(epsg=4326)

gdf.head()

,cadastralDistrictsCode,category,descr,externalKey,geom_data_id,label,options,subcategory,system_info,geometry
0,78,36368,78:32:0001676:3999,78:32:0001676:3999,426354188,78:32:0001676:3999,"{'area': None, 'status': 'Учтенный', 'cad_num'...",5,"{'updated': '2025-02-05T17:26:07.668605', 'ins...","POLYGON ((3373004.44 8380148.621, 3373008.658 ..."
1,78,36368,78:32:0001679:3,78:32:0001679:3,155135205,78:32:0001679:3,"{'area': None, 'status': 'Ранее учтенный', 'ca...",5,"{'updated': '2025-02-05T16:16:52.988075', 'ins...","POLYGON ((3374635.7 8380171.432, 3374655.255 8..."
2,78,36368,78:06:0002079:1592,78:06:0002079:1592,155131697,78:06:0002079:1592,"{'area': None, 'status': 'Учтенный', 'cad_num'...",5,"{'updated': '2025-02-05T18:16:41.759469', 'ins...","POLYGON ((3369792.195 8385162.551, 3369749.277..."
3,78,36368,78:06:0002038:4121,78:06:0002038:4121,155131394,78:06:0002038:4121,"{'area': None, 'status': 'Учтенный', 'cad_num'...",5,"{'updated': '2025-02-05T16:16:51.937561', 'ins...","POLYGON ((3369803.403 8387192.435, 3369811.12 ..."
4,78,36368,78:31:0001282:3199,78:31:0001282:3199,135819688,78:31:0001282:3199,"{'area': None, 'status': 'Учтенный', 'cad_num'...",5,"{'updated': '2025-02-05T16:02:05.245654', 'ins...","POLYGON ((3378556.552 8385647.343, 3378591.4 8..."


In [ ]:
# ===========================================================
#  Async WMS scraper  •  resume-from-pixel 159 900
# ===========================================================

# !pip install --upgrade aiohttp shapely geopandas pyarrow tqdm nest_asyncio

import asyncio, hashlib, random
from pathlib import Path

import aiohttp, geopandas as gpd
import pandas as pd
from shapely.geometry import shape
from tqdm.auto import tqdm
from json.decoder import JSONDecodeError
from re import search
# -- Jupyter-patch (в .py-файле не нужен) --------------------
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

# ------------------- ПАРАМЕТРЫ ------------------------------
URL  = "https://nspd.gov.ru/api/aeggis/v4/36048/wms"
WIDTH, HEIGHT = 512, 512
BBOX  = [3130860.6785608195, 8140237.7642581295,
         3757032.8142729835, 8766409.899970293]

CONCURRENCY   = 64
TIMEOUT_SEC   = 15
RETRIES       = 3
BACKOFF_BASE  = 0.7

FLUSH_EVERY   = 30_000
OUT_DIR       = Path("parsed_parts").resolve()
OUT_DIR.mkdir(exist_ok=True)

LAST_IDX_PROCESSED = 159_899   # ← ваш последний обработанный индекс
# ------------------------------------------------------------

BASE_PARAMS = {
    "REQUEST": "GetFeatureInfo",
    "QUERY_LAYERS": "36048",
    "SERVICE": "WMS",
    "VERSION": "1.3.0",
    "FORMAT": "image/png",
    "STYLES": "",
    "TRANSPARENT": "true",
    "LAYERS": "36048",
    "INFO_FORMAT": "application/json",
    "FEATURE_COUNT": "10",
    "CRS": "EPSG:3857",
    "WIDTH": str(WIDTH),
    "HEIGHT": str(HEIGHT),
    "BBOX": ",".join(map(str, BBOX)),
}

# ---------- Сканирует OUT_DIR и возвращает (max + 1) для part_XXXX.parquet. ----
def next_part_index() -> int:
    ix = -1
    for p in OUT_DIR.glob("part_*.parquet"):
        m = search(r"part_(\d{4})\.parquet", p.name)
        if m:
            ix = max(ix, int(m.group(1)))
    return ix + 1            # если файлов нет → вернёт 0

# ---------- восстановление id из уже записанных частей -----
def restore_seen_ids():
    ids = set()
    for p in sorted(OUT_DIR.glob("part_*.parquet")):
        # читаем ТОЛЬКО столбец geom_data_id, геометрию не трогаем
        df = pd.read_parquet(p, columns=["geom_data_id"])
        ids |= set(df["geom_data_id"].dropna().astype(str))
    return ids

# ---------- утилиты ----------------------------------------
def feat_id(feat) -> str:
    gid = feat["properties"].get("geom_data_id")
    if gid is not None:
        return str(gid)
    geom_bytes  = shape(feat["geometry"]).wkb
    props_bytes = str(sorted(feat["properties"].items())).encode()
    return hashlib.md5(geom_bytes + props_bytes).hexdigest()

async def fetch_pixel(session, i, j):
    params = BASE_PARAMS | {"I": str(i), "J": str(j)}
    delay = BACKOFF_BASE * (0.7 + random.random() * 0.6)
    for attempt in range(1, RETRIES + 1):
        try:
            async with session.get(URL, params=params, ssl=False) as resp:
                if resp.status != 200:
                    return []
                try:
                    data = await resp.json()
                except (aiohttp.ContentTypeError, JSONDecodeError):
                    await resp.read()
                    return []
                return data.get("features", [])
        except (asyncio.TimeoutError,
                aiohttp.ClientError,
                asyncio.CancelledError):
            if attempt == RETRIES:
                return []
            await asyncio.sleep(delay)
            delay *= 2

# ---------- writer -----------------------------------------
async def writer(wq: asyncio.Queue):
    buffer = []
    part_ix = next_part_index()    
    
    async def flush():
        nonlocal part_ix, buffer
        if not buffer:
            return
        attrs, geoms = zip(*buffer)
        gdf = gpd.GeoDataFrame(list(attrs), geometry=list(geoms), crs="EPSG:3857")
        out_path = OUT_DIR / f"part_{part_ix:04d}.parquet"
        gdf.to_parquet(out_path, engine="pyarrow", index=False, compression=None)
        print(f"💾  part {part_ix:04d} → {len(gdf):,} объектов (Parquet)")
        buffer.clear(); part_ix += 1
        
    while True:
        item = await wq.get()
        if item is None:
            await flush(); wq.task_done(); return
        buffer.append(item); wq.task_done()
        if len(buffer) >= FLUSH_EVERY:
            await flush()

# ---------- consumer ---------------------------------------
async def consumer(pq, wq, session, seen, pbar):
    while True:
        item = await pq.get()
        if item is None:
            pq.task_done(); return
        i, j = item
        try:
            for feat in await fetch_pixel(session, i, j):
                fid = feat_id(feat)
                if fid in seen:
                    continue
                seen.add(fid)
                wq.put_nowait((feat["properties"], shape(feat["geometry"])))
        except Exception as e:
            print(f"⚠️  pixel({i},{j}) → {type(e).__name__}: {e}")
        finally:
            pq.task_done(); pbar.update(1)

# ---------- main -------------------------------------------
async def main():
    total_px = WIDTH * HEIGHT
    start_idx = LAST_IDX_PROCESSED
    start_row, start_col = divmod(start_idx, WIDTH)

    pq, wq = asyncio.Queue(), asyncio.Queue()
    seen_ids = restore_seen_ids()
    print(f"🔄  Уже сохранённых геометрий: {len(seen_ids):,}")

    connector = aiohttp.TCPConnector(limit=CONCURRENCY, force_close=False)
    timeout   = aiohttp.ClientTimeout(TIMEOUT_SEC)

    async with aiohttp.ClientSession(connector=connector, timeout=timeout) as sess:
        with tqdm(total=total_px, desc="Скачиваем", initial=start_idx) as pbar:
            writer_task = asyncio.create_task(writer(wq))
            workers = [asyncio.create_task(consumer(pq, wq, sess, seen_ids, pbar))
                       for _ in range(CONCURRENCY)]

            # кладём оставшиеся пиксели
            for j in range(start_row, HEIGHT):
                i0 = start_col if j == start_row else 0
                for i in range(i0, WIDTH):
                    await pq.put((i, j))

            await pq.join()
            for _ in workers:
                await pq.put(None)
            await asyncio.gather(*workers)
            await wq.put(None); await wq.join(); await writer_task

    print(f"🎉  Завершено. Всего уникальных объектов: {len(seen_ids):,}")

# ---- запуск (в ноутбуке) -----------------------------------
await main()

# для скрипта:
# if __name__ == "__main__":
#     asyncio.run(main())


Скачиваем:   0%|          | 0/262144 [00:00<?, ?it/s]

📈 Скачано уникальных объектов: 5,000
📈 Скачано уникальных объектов: 10,000
📈 Скачано уникальных объектов: 15,000
📈 Скачано уникальных объектов: 20,000
💾  part 0000 → 20,000 объектов (Parquet)
📈 Скачано уникальных объектов: 25,000
📈 Скачано уникальных объектов: 30,000
📈 Скачано уникальных объектов: 35,000
📈 Скачано уникальных объектов: 40,000
💾  part 0001 → 20,000 объектов (Parquet)
📈 Скачано уникальных объектов: 45,000
📈 Скачано уникальных объектов: 50,000
📈 Скачано уникальных объектов: 55,000
📈 Скачано уникальных объектов: 60,000
💾  part 0002 → 20,000 объектов (Parquet)
📈 Скачано уникальных объектов: 65,000
📈 Скачано уникальных объектов: 70,000
📈 Скачано уникальных объектов: 75,000
📈 Скачано уникальных объектов: 80,000
💾  part 0003 → 20,000 объектов (Parquet)
📈 Скачано уникальных объектов: 85,000
📈 Скачано уникальных объектов: 90,000
📈 Скачано уникальных объектов: 95,000
📈 Скачано уникальных объектов: 100,000
💾  part 0004 → 20,000 объектов (Parquet)
📈 Скачано уникальных объектов: 105,